# Step 5 — the entropy gate: does GRPO have a gradient to work with?

**August plan, week 1, `local/tasks/plan-accion.md`.** Not a rung: the ladder is closed
([[august-plan-closes-the-ladder]]). This notebook produces two numbers and states a
pre-declared verdict. It trains nothing.

## Pre-registration — declared BEFORE the run

| number | meaning | kills the phase if |
|---|---|---|
| `zero_advantage_frac` | share of questions where all k rollouts earn the **same reward** | **>= 0.60** |
| `pass@k - greedy` | ceiling sampling can reach above greedy | **< +0.05** |

The first says whether there is a **gradient**. The second says how much **headroom**.
Both come out of the same pass. **Reported per `answer_format` — `number`, `fo_class` and
`binary` separately — and never averaged into one figure.**

🔴 **Checkpoint is A2 ep3 (`21_lr_2e4_v1/checkpoint-2703`), not rung 06.** Measuring on a
checkpoint we are about to replace was rung 18's mistake.
🔴 **Questions come from TRAIN**, not from val: this measures the policy's own sampling
behaviour where GRPO would actually run, and spends no val signal.

## What it decides

Green → phase C (GRPO with a set-F1 reward) is alive and goes to week 3.
Red → phase C dies and the pre-decided **Plan B is rung 22 (loss-mass) rebased on A2, the
next day** — its `compute_loss_func` hook is built and its OFF invariant is verified
(`src/frame/loss.py:212`).

In [ ]:
# --- parameters (papermill) -----------------------------------------------------
# 🔴 Comments go ABOVE the assignment, never on the same line: papermill's parameter
# parser silently skips a line it cannot parse, `-p` is then ignored, and the notebook
# runs on its default. Measured 2026-08-05 on SMOKE and TEMPERATURE.

RUN = "21_lr_2e4_v1"            # A2
CKPT_NAME = "checkpoint-2703"   # epoch 3
K = 8
N_Q = 600
# GRPO samples at T=1; a lower T would measure a policy that is not the one being trained
TEMPERATURE = 1.0
SEED = 42
# True -> 24 questions, k=4. Flip to False only AFTER the smoke reads.
SMOKE = True
DATA_ROOT = "/workspace/orena-data"

In [ ]:
# --- derived (MUST live BELOW the parameters cell — the rung-16 papermill trap) --
import time, json, gc
from pathlib import Path
import numpy as np, pandas as pd

REPO = Path.cwd()
while not (REPO / "src" / "frame").is_dir():
    assert REPO != REPO.parent, "run me from inside the repo"
    REPO = REPO.parent
import sys
sys.path.insert(0, str(REPO / "src"))

EXP     = REPO / "experiments" / "10-self-consistency"
RUN_DIR = EXP / "runs" / f"step5_entropy_{RUN}"
TAG     = "smoke" if SMOKE else "full"
OUT     = RUN_DIR / TAG
OUT.mkdir(parents=True, exist_ok=True)

if SMOKE:
    N_Q, K = 24, 4

# The QA parquets live in different places on the pod and on a laptop. Resolve by LOOKING,
# and fail loudly: an eval on an empty data root is the classic silent zero.
DATA_ROOT = next(
    (d for d in (Path(DATA_ROOT), REPO / "external_data" / "orena-data")
     if (d / "heico" / "data" / "frame" / "train.parquet").exists()), None)
assert DATA_ROOT is not None, "no frame/train.parquet found — pull the QA parquets"
print(f"OK    data_root  {DATA_ROOT}")
print(f"OK    out        {OUT}")
print(f"      N_Q={N_Q}  K={K}  T={TEMPERATURE}  smoke={SMOKE}")

In [ ]:
# --- the train questions, stratified by answer_format --------------------------
from frame.config import BaselineConfig
from frame.data import FrameProvider, load_frame_items
from frame import split as fsplit

cfg_load = BaselineConfig(data_root=DATA_ROOT, out_dir=OUT, run_name=TAG, seed=SEED)
items = [it for it in load_frame_items(cfg_load, splits=("train",))]
print(f"      train items: {len(items):,}")

WANT = ("number", "fo_class", "binary")
pool = pd.DataFrame({"i": np.arange(len(items)),
                     "answer_format": [fsplit._answer_format(it) for it in items]})
pool = pool[pool.answer_format.isin(WANT)]
assert not pool.empty, f"no items in {WANT} — check split._answer_format's vocabulary"

# Equal-ish per format: the gate is read PER FORMAT, so a proportional sample would
# leave the rarest format with a denominator too small to read.
per = max(1, N_Q // len(WANT))
picked = pd.concat([g.sample(n=min(per, len(g)), random_state=SEED)
                    for _, g in pool.groupby("answer_format")])
sel = [items[i] for i in picked.i.tolist()]
print(picked.answer_format.value_counts().to_string())

In [ ]:
# --- the checkpoint: A2 ep3, merged -------------------------------------------
# The pod carries FOUR checkouts and rung 21's runs/ lives in whichever one launched it,
# so resolve by LOOKING (same discipline as DATA_ROOT above) instead of assuming REPO.
CAND = [Path(p) / "experiments/21-recipe-sweep/runs" / RUN / "merged" / CKPT_NAME
        for p in ("/workspace/repo", "/workspace/repo_leo", "/workspace/repo_rodri",
                  "/workspace/repo_yyy", str(REPO))]
MERGED = next((d for d in CAND if d.is_dir() and any(d.iterdir())), None)
assert MERGED is not None, (
    "merged checkpoint not found in any checkout:\n  " + "\n  ".join(map(str, CAND)) +
    "\nMerge it with rung 02's merge_checkpoint "
    "(experiments/02-lora-sft/_models/lora_sft_train.py:254) from the adapter — do NOT "
    "re-merge inside this notebook, a merge is a heavy artifact owned by its own run.")
print(f"OK    merged  {MERGED}")

In [ ]:
# --- generate: greedy + K sampled rollouts per question ------------------------
from frame.engine import QwenFrameEngine

t0 = time.perf_counter()
base = dict(data_root=DATA_ROOT, model_path=MERGED, out_dir=OUT, run_name=TAG,
            max_pixels=1280*720, seed=SEED)

# `.load()` is EXPLICIT: the constructor only sets model/processor to None, and the engine
# catches every inference exception and returns "Inference Error: ..." k times rather than
# raising. Forgetting load() therefore produces a notebook that completes GREEN on 100%
# garbage — measured on this notebook's first smoke, 2026-08-05.
def run_arm(cfg, sampled):
    prov = FrameProvider(cfg)
    eng = QwenFrameEngine(cfg)
    eng.load()
    assert eng.model is not None and eng.processor is not None, "engine.load() did not load"
    out = []
    for it in sel:
        prov.ensure_reader(it)
        img = prov.get_frame(it)
        out.append(eng.predict_samples(img, it.request.question) if sampled
                   else eng.predict(img, it.request.question))
    eng.unload(); del eng; gc.collect()
    return out

cfg_greedy = BaselineConfig(**base)
assert cfg_greedy.n_samples == 1 and cfg_greedy.answer_postprocess is None, \
    "greedy arm must be the untouched inference path"
greedy = run_arm(cfg_greedy, sampled=False)

cfg_k = BaselineConfig(**base, n_samples=K, temperature=TEMPERATURE)
samples = run_arm(cfg_k, sampled=True)

# GATE — the engine NEVER raises, so a dead model reads as a finished run.
flat = list(greedy) + [s for row in samples for s in row]
bad = [a for a in flat if a.startswith("Inference Error:")]
assert not bad, (
    f"GATE — {len(bad)}/{len(flat)} generations are engine errors, e.g. {bad[0]!r}. "
    "Every number below would be computed on garbage.")
assert all(len(s) == K for s in samples), "predict_samples did not return K candidates"
print(f"      generated in {time.perf_counter()-t0:.0f}s  ({len(flat)} generations, 0 errors)")

In [ ]:
# --- score: the SDK verifier, K+1 passes, judge loaded ONCE ---------------------
# RULES EVAL: score ONLY via the vendor path. A local string comparison here would make
# the gate measure our parser instead of the reward GRPO would actually receive.
from focus.data.data_models import Response
from focus.evaluation.evaluator import Evaluator
from focus.evaluation.judges import TransformersJudge

# FrameItem already carries the SDK objects — no re-parsing.
requests   = [it.request   for it in sel]
references = [it.reference for it in sel]
for ref in references:
    ref.ood = ref.qID.split("__", 1)[0] == "heico"
qids = [r.qID for r in requests]

judge = TransformersJudge(model_name=cfg_k.judge_model, device=cfg_k.device)
evaluator = Evaluator(judges=[judge], seed=SEED)

def score(answers, tag):
    responses = [Response(qID=q, content=a, latency=0.0) for q, a in zip(qids, answers)]
    d = OUT / tag; d.mkdir(exist_ok=True)
    res, _ = evaluator.run(requests=requests, references=references,
                           responses=responses, output_dir=d, track=None)
    return res.set_index("qID")["correctness"].astype(bool)

greedy_correct  = score(greedy, "greedy")
samples_correct = [score([s[j] for s in samples], f"s{j}") for j in range(K)]

del judge, evaluator; gc.collect()
print(f"OK    scored {K+1} passes over {len(sel)} questions")

In [ ]:
# --- the two numbers, per format, NOT averaged ---------------------------------
from frame import metrics, vote

# fo_class is scored as a SET, so its key must use the SDK's own class vocabulary —
# the same `_load_fotype()` the scorer uses, never a hand-typed list.
valid_lower = {n.lower(): n for n in metrics._load_fotype().names()}
keys = {
    "number":   vote.key_number,
    "binary":   vote.key_binary,
    "fo_class": lambda t: vote.key_fo_class(t, valid_lower),
}

records = pd.DataFrame({
    "qID": qids,
    "answer_format": [fsplit._answer_format(it) for it in sel],
    "samples": samples,
    "samples_correct": [[bool(sc.loc[q]) for sc in samples_correct] for q in qids],
    "greedy_correct": [bool(greedy_correct.loc[q]) for q in qids],
})

pq = vote.per_question(records, keys=keys)
pq.to_csv(OUT / "per_question.csv", index=False)     # 🔑 the artifact rung 10 never had
gate = vote.gate_by_format(pq)
gate.to_csv(OUT / "gate_by_format.csv", index=False)
print(gate.to_string(index=False))

In [ ]:
# --- verdict against the PRE-DECLARED thresholds -------------------------------
ZERO_ADV_KILL = 0.60
HEADROOM_KILL = 0.05

verdict = {}
for r in gate.itertuples(index=False):
    dead = (r.zero_advantage_frac >= ZERO_ADV_KILL) or (r.headroom < HEADROOM_KILL)
    why = []
    if r.zero_advantage_frac >= ZERO_ADV_KILL:
        why.append(f"zero_advantage {r.zero_advantage_frac:.3f} >= {ZERO_ADV_KILL}")
    if r.headroom < HEADROOM_KILL:
        why.append(f"headroom {r.headroom:+.3f} < +{HEADROOM_KILL}")
    verdict[r.answer_format] = {"n": int(r.n), "dead": bool(dead),
                                "why": why or ["passes both"]}
    print(f"{r.answer_format:9s} n={r.n:4d}  {'🔴 DEAD' if dead else '🟢 ALIVE'}  {'; '.join(why) or 'passes both'}")

(OUT / "verdict.json").write_text(json.dumps(verdict, indent=2))
print("\n🔴 Phase C dies if EVERY format is dead. A single live format keeps it, scoped to that format.")
print("   Plan B if dead: rung 22 (loss-mass) rebased on A2, the next day.")

## Reading it

- **Per format, never pooled.** A pooled figure would let a live `fo_class` hide a dead
  `number`, and `number` is 80.4% of `aggregation`.
- **`zero_advantage` is defined on the REWARD, not on the answer text** (`vote.py`). Two
  rollouts that differ textually but are both wrong give identical rewards and therefore no
  gradient. Measuring answer-identity would report more usable signal than GRPO can see —
  wrong in the optimistic direction, which for a kill gate is the dangerous one.
- **`n_unparsed`** is carried per question rather than dropped: an unparseable rollout is
  scored 0 by the SDK, so it belongs in the denominator.
- **This gate does not license GRPO.** It only says the gradient is not identically zero.
  Week 2's smoke (`s_per_it`, peak MiB, observed zero-advantage rate, illegal-token rate)
  is still owed before any long run is scheduled.